# Google Compute Engine 인스턴스 생성 및 정책 설정

이 주피터 노트북은 Google Cloud SDK(`gcloud`)를 사용하여 다음 리소스를 생성하고 구성합니다:
1. **Compute Engine 인스턴스 생성**: `instance-20260914-054827` (`e2-medium`, Debian 13, 10GB pd-balanced)
2. **Google Cloud Ops Agent 정책 설정**: `config.yaml` 생성 및 정책 등록
3. **디스크 스냅샷 스케줄 정책 생성 및 연결**: 매일 23:00 자동 스냅샷 (14일 보관)

> **참고**: 노트북 셀에서 셸 명령어를 실행하기 위해 맨 위에 `%%bash` 매직 커맨드를 사용합니다.

## 1. 전체 스크립트 일괄 실행
아래 셀을 실행하면 인스턴스 생성부터 Ops Agent 정책 등록, 스냅샷 스케줄 연결까지 한 번에 순차적으로 실행됩니다.

In [ ]:
%%bash
gcloud compute instances create instance-20260915-143300 \
    --project=iceu-songpa25 \
    --zone=us-central1-a \
    --machine-type=e2-medium \
    --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default \
    --metadata=enable-osconfig=TRUE \
    --maintenance-policy=MIGRATE \
    --provisioning-model=STANDARD \
    --service-account=910486776157-compute@developer.gserviceaccount.com \
    --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
    --create-disk=auto-delete=yes,boot=yes,device-name=instance-20260914-054827,image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced \
    --no-shielded-secure-boot \
    --shielded-vtpm \
    --shielded-integrity-monitoring \
    --labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud \
    --reservation-affinity=any \
&& \
printf 'agentsRule:\n  packageState: installed\n  version: latest\ninstanceFilter:\n  inclusionLabels:\n  - labels:\n      goog-ops-agent-policy: v2-template-1-7-0\n' > config.yaml \
&& \
gcloud compute instances ops-agents policies create goog-ops-agent-v2-template-1-7-0-us-central1-a \
    --project=iceu-songpa25 \
    --zone=us-central1-a \
    --file=config.yaml \
&& \
gcloud compute resource-policies create snapshot-schedule default-schedule-1 \
    --project=iceu-songpa25 \
    --region=us-central1 \
    --max-retention-days=14 \
    --on-source-disk-delete=keep-auto-snapshots \
    --daily-schedule \
    --start-time=23:00 \
&& \
gcloud compute disks add-resource-policies instance-20260914-054827 \
    --project=iceu-songpa25 \
    --zone=us-central1-a \
    --resource-policies=projects/iceu-songpa25/regions/us-central1/resourcePolicies/default-schedule-1


## 2. 단계별 실행 (선택 사항)
과정을 단계별로 분리하여 실행하고 각 결과를 확인할 수 있습니다.

### Step 2-1. Compute Engine 인스턴스 생성

In [ ]:
%%bash
gcloud compute instances create instance-20260914-054827 \
    --project=iceu-songpa25 \
    --zone=us-central1-a \
    --machine-type=e2-medium \
    --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default \
    --metadata=enable-osconfig=TRUE \
    --maintenance-policy=MIGRATE \
    --provisioning-model=STANDARD \
    --service-account=910486776157-compute@developer.gserviceaccount.com \
    --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
    --create-disk=auto-delete=yes,boot=yes,device-name=instance-20260914-054827,image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced \
    --no-shielded-secure-boot \
    --shielded-vtpm \
    --shielded-integrity-monitoring \
    --labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud \
    --reservation-affinity=any


### Step 2-2. Cloud Ops Agent 정책 설정 (config.yaml 생성 및 적용)

In [ ]:
%%bash
printf 'agentsRule:\n  packageState: installed\n  version: latest\ninstanceFilter:\n  inclusionLabels:\n  - labels:\n      goog-ops-agent-policy: v2-template-1-7-0\n' > config.yaml

gcloud compute instances ops-agents policies create goog-ops-agent-v2-template-1-7-0-us-central1-a \
    --project=iceu-songpa25 \
    --zone=us-central1-a \
    --file=config.yaml


### Step 2-3. 자동 스냅샷 스케줄 정책 생성 및 디스크 연결

In [ ]:
%%bash
gcloud compute resource-policies create snapshot-schedule default-schedule-1 \
    --project=iceu-songpa25 \
    --region=us-central1 \
    --max-retention-days=14 \
    --on-source-disk-delete=keep-auto-snapshots \
    --daily-schedule \
    --start-time=23:00

gcloud compute disks add-resource-policies instance-20260914-054827 \
    --project=iceu-songpa25 \
    --zone=us-central1-a \
    --resource-policies=projects/iceu-songpa25/regions/us-central1/resourcePolicies/default-schedule-1


## 3. 결과 확인
생성된 인스턴스의 상태 및 세부 정보를 확인합니다.

In [ ]:
%%bash
gcloud compute instances list --filter="name=instance-20260914-054827"


---
# 4. GCP 최저가 리전 Top 3 비교 및 최적 리전 생성

동일한 인스턴스 사양(`e2-medium`, 2 vCPU / 4GB RAM, 10GB `pd-balanced`, Premium Network) 기준, 전 세계 GCP 리전 중 가장 저렴한 Top 3 리전 분석 결과입니다.

### 💰 최저가 리전 Top 3 비교
| 순위 | 리전 (Region) | 위치 | e2-medium (월) | 디스크 (10GB) | 총 예상 비용 (월) | 한국 대비 절감률 / 특징 |
| :---: | :--- | :--- | :---: | :---: | :---: | :--- |
| **1위 (공동)** | **`us-central1`** | 미국 아이오와 | $24.46 ($0.0335/h) | $1.00 | **$25.46** (~3.4만원) | 기존 설정 리전, 전 세계 최저가 |
| **2위 (공동)** | **`us-west1`** | 미국 오레곤 | $24.46 ($0.0335/h) | $1.00 | **$25.46** (~3.4만원) | **추천: 최저가 + 태평양 직결로 한국 지연시간 최단** |
| **3위 (공동)** | **`us-east1`** | 미국 사우스캐롤라이나 | $24.46 ($0.0335/h) | $1.00 | **$25.46** (~3.4만원) | 전 세계 최저가 티어 |
| *참고* | `asia-northeast3` | 대한민국 서울 | $30.37 ($0.0416/h) | $1.20 | **$31.57** (~4.2만원) | 국내 최저 지연시간이나 약 24% 비쌈 |

> **💡 핵심 분석**:
> - `us-central1`, `us-west1`, `us-east1` (및 `us-east5`) 3곳 모두 월 **$25.46**로 전 세계 공동 1위 최저가입니다.
> - 기존 코드의 `us-central1`은 이미 최저가였으며, 한국에서 원격 접속/SSH/개발을 진행할 때는 태평양 해저 광케이블을 직결하는 **`us-west1` (오레곤)** 리전이 네트워크 응답 속도(Latency ~130ms) 면에서 가장 우수합니다.

### 4-1. 최저가 추천 리전(`us-west1-b`, 오레곤) 인스턴스 일괄 생성
동일한 옵션(e2-medium, 10GB pd-balanced, Ops Agent, 일일 스냅샷)을 유지하며 가장 지연시간이 낮은 최저가 리전 `us-west1-b`에 배포합니다.

In [ ]:
%%bash
# 대상 리전 및 영역 설정 (최저가 + 한국 접속 지연시간 최단)
TARGET_PROJECT="iceu-songpa25"
TARGET_REGION="us-west1"
TARGET_ZONE="us-west1-b"
INSTANCE_NAME="instance-cheapest-$(date +%Y%m%d-%H%M%S)"

echo "=== 최저가 리전(${TARGET_ZONE}) 인스턴스 생성 시작: ${INSTANCE_NAME} ==="

gcloud compute instances create ${INSTANCE_NAME} \
    --project=${TARGET_PROJECT} \
    --zone=${TARGET_ZONE} \
    --machine-type=e2-medium \
    --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default \
    --metadata=enable-osconfig=TRUE \
    --maintenance-policy=MIGRATE \
    --provisioning-model=STANDARD \
    --service-account=910486776157-compute@developer.gserviceaccount.com \
    --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
    --create-disk=auto-delete=yes,boot=yes,device-name=${INSTANCE_NAME},image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced \
    --no-shielded-secure-boot \
    --shielded-vtpm \
    --shielded-integrity-monitoring \
    --labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud \
    --reservation-affinity=any \
&& \
printf 'agentsRule:\n  packageState: installed\n  version: latest\ninstanceFilter:\n  inclusionLabels:\n  - labels:\n      goog-ops-agent-policy: v2-template-1-7-0\n' > config.yaml \
&& \
gcloud compute instances ops-agents policies create goog-ops-agent-v2-template-1-7-0-${TARGET_ZONE} \
    --project=${TARGET_PROJECT} \
    --zone=${TARGET_ZONE} \
    --file=config.yaml \
&& \
gcloud compute resource-policies create snapshot-schedule default-schedule-${TARGET_REGION} \
    --project=${TARGET_PROJECT} \
    --region=${TARGET_REGION} \
    --max-retention-days=14 \
    --on-source-disk-delete=keep-auto-snapshots \
    --daily-schedule \
    --start-time=23:00 \
&& \
gcloud compute disks add-resource-policies ${INSTANCE_NAME} \
    --project=${TARGET_PROJECT} \
    --zone=${TARGET_ZONE} \
    --resource-policies=projects/${TARGET_PROJECT}/regions/${TARGET_REGION}/resourcePolicies/default-schedule-${TARGET_REGION}

echo "=== 생성 확인 ==="
gcloud compute instances list --filter="name=${INSTANCE_NAME}"


### 4-2. 파이썬(Python)으로 최저가 리전 선택 및 배포하기
노트북 내에서 변수를 수정하여 원하는 최저가 리전(`us-west1`, `us-central1`, `us-east1`)을 선택해 배포할 수 있습니다.

In [ ]:
import subprocess
import time

# [설정] 최저가 리전 중 선택: 'us-west1'(오레곤 - 지연시간 최단), 'us-central1'(아이오와), 'us-east1'(사우스캐롤라이나)
SELECTED_REGION = "us-west1"
SELECTED_ZONE = f"{SELECTED_REGION}-b"
PROJECT_ID = "iceu-songpa25"
INSTANCE_NAME = f"instance-{SELECTED_REGION}-{int(time.time())}"

print(f"🚀 최저가 리전 [{SELECTED_ZONE}]에 인스턴스 배포 시작: {INSTANCE_NAME}")

create_cmd = [
    "gcloud", "compute", "instances", "create", INSTANCE_NAME,
    f"--project={PROJECT_ID}",
    f"--zone={SELECTED_ZONE}",
    "--machine-type=e2-medium",
    "--network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default",
    "--metadata=enable-osconfig=TRUE",
    "--maintenance-policy=MIGRATE",
    "--provisioning-model=STANDARD",
    "--service-account=910486776157-compute@developer.gserviceaccount.com",
    "--scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append",
    f"--create-disk=auto-delete=yes,boot=yes,device-name={INSTANCE_NAME},image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced",
    "--no-shielded-secure-boot",
    "--shielded-vtpm",
    "--shielded-integrity-monitoring",
    "--labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud",
    "--reservation-affinity=any"
]

res = subprocess.run(create_cmd, capture_output=True, text=True)
if res.returncode == 0:
    print(f"✅ 인스턴스 {INSTANCE_NAME} 생성 성공!")
    print(res.stdout)
else:
    print(f"❌ 생성 실패: {res.stderr}")


---
# 5. 리소스 정리 및 과금 방지 (Cleanup & Billing Guard)

실습 및 테스트가 끝난 후, 인스턴스와 디스크 및 관련 정책을 완전히 삭제하여 **익일 과금을 방지**하는 단계입니다.

1. **실습 리소스 삭제**: 생성된 인스턴스, Ops Agent 정책, 스냅샷 스케줄 삭제
2. **유휴 자원 일괄 정리**: 혹시 남아있을 수 있는 고아(미연결) 디스크 및 잔여 VM 정리
3. **과금 방지 하네스 점검**: 프로젝트 내 10대 과금 유발 요소가 완전히 0건인지 최종 검증

### 5-1. 실습 생성 리소스 일괄 삭제
노트북에서 생성했던 인스턴스, Ops Agent 정책, 스냅샷 스케줄 정책 및 임시 설정 파일을 안전하게 삭제합니다.

In [2]:
%%bash
PROJECT_ID="iceu-songpa25"
echo "🚀 [1/4] Compute Engine 인스턴스 삭제 시작..."

# 1. us-central1-a 기본 인스턴스 삭제
gcloud compute instances delete instance-20260914-054827 \
    --zone=us-central1-a \
    --project=${PROJECT_ID} \
    --quiet 2>/dev/null || echo "- instance-20260914-054827 이미 삭제되었거나 없음"

# 2. 최저가 리전 인스턴스(instance-cheapest-* 또는 instance-us-*) 일괄 검색 및 삭제
INSTANCES=$(gcloud compute instances list --project=${PROJECT_ID} --filter="name ~ 'instance-cheapest.*|instance-us-.*'" --format="value(name,zone)")
if [ -n "$INSTANCES" ]; then
  while read -r name zone; do
    if [ -n "$name" ]; then
      echo "- 인스턴스 삭제: $name (Zone: $zone)"
      gcloud compute instances delete "$name" --zone="$zone" --project=${PROJECT_ID} --quiet
    fi
  done <<< "$INSTANCES"
else
  echo "- 최저가 테스트 인스턴스 없음"
fi

echo "🚀 [2/4] Ops Agent 정책 삭제..."
gcloud compute instances ops-agents policies delete goog-ops-agent-v2-template-1-7-0-us-central1-a --zone=us-central1-a --project=${PROJECT_ID} --quiet 2>/dev/null || true
gcloud compute instances ops-agents policies delete goog-ops-agent-v2-template-1-7-0-us-west1-b --zone=us-west1-b --project=${PROJECT_ID} --quiet 2>/dev/null || true

echo "🚀 [3/4] 스냅샷 스케줄 정책 삭제..."
gcloud compute resource-policies delete default-schedule-1 --region=us-central1 --project=${PROJECT_ID} --quiet 2>/dev/null || true
gcloud compute resource-policies delete default-schedule-us-west1 --region=us-west1 --project=${PROJECT_ID} --quiet 2>/dev/null || true

echo "🚀 [4/4] 로컬 설정 임시 파일 삭제..."
rm -f config.yaml

echo "✅ 주요 실습 리소스 삭제 완료!"


🚀 [1/4] Compute Engine 인스턴스 삭제 시작...


ERROR: (gcloud.compute.instances.list) Term operand expected [name ~ *HERE* (instance-cheapest.*|instance-us-.*)].


- 최저가 테스트 인스턴스 없음
🚀 [2/4] Ops Agent 정책 삭제...
🚀 [3/4] 스냅샷 스케줄 정책 삭제...
🚀 [4/4] 로컬 설정 임시 파일 삭제...
✅ 주요 실습 리소스 삭제 완료!


### 5-2. 혹시 남아있는 유휴 영구 디스크(Unattached Disks) 정리
VM 삭제 시 부팅 디스크가 자동 삭제되지 않았거나 별도로 생성된 고아 디스크가 있다면 지속적인 스토리지 요금이 발생합니다. 아래 셀은 인스턴스에 연결되지 않은 유휴 디스크를 찾아 삭제합니다.

In [ ]:
%%bash
echo "🔍 연결되지 않은 유휴 영구 디스크 검색 중..."
UNATTACHED_DISKS=$(gcloud compute disks list --filter="-users:*" --format="value(name,zone)")

if [ -z "$UNATTACHED_DISKS" ]; then
  echo "✅ 유휴 영구 디스크가 없습니다. (안전)"
else
  echo "⚠️ 미연결 디스크 발견, 삭제를 진행합니다:"
  while read -r name zone; do
    if [ -n "$name" ]; then
      echo "- 디스크 삭제 중: $name ($zone)"
      gcloud compute disks delete "$name" --zone="$zone" --quiet
    fi
  done <<< "$UNATTACHED_DISKS"
  echo "✅ 유휴 디스크 정리 완료!"
fi


### 5-3. 🛡️ 과금 방지 하네스 최종 전수 점검 (필수 실행)
VM, 디스크, 미사용 고정 IP, 스냅샷, 스토리지 버킷 등 **10대 주요 과금 영역**에 잔여 자원이 없는지 최종 확인합니다.
- **모든 리소스 0개 (`✅ 안전`)**: 안심하고 작업을 종료하셔도 됩니다.
- **미삭제 자원 발견 시 (`⚠️ 경고`)**: 화면에 즉시 삭제 명령어가 출력됩니다.

In [3]:
%%bash
# 프로젝트에 구축된 과금 방지 점검 하네스 실행
./check_resources.sh


## 🛡️ GCP 잔여 자원 및 과금 위험 점검 보고서
- **점검 시각**: `2026-09-14 15:29:34`
- **대상 프로젝트**: `iceu-songpa25`
- **상태 요약**: ✅ **안전: 과금 유발 잔여 리소스 없음 (0건)**

| 리소스 분류 | 리소스 항목 | 상태 / 수량 | 과금 위험도 |
| :--- | :--- | :---: | :---: |
| Compute | Compute Engine (VM 인스턴스) | 0대 | 🟢 안전 |
| Storage | Persistent Disks (영구 디스크) | 0개 | 🟢 안전 |
| Network | Static External IPs (고정 외부 IP) | 0개 | 🟢 안전 |
| Storage | Disk Snapshots (디스크 스냅샷) | 0개 | 🟢 안전 |
| Compute | Custom Images (사용자 정의 머신 이미지) | 0개 | 🟢 안전 |
| Network | Forwarding Rules (부하 분산기 / 로드밸런서) | 0개 | 🟢 안전 |
| Network | Cloud Routers & NAT | 0개 | 🟢 안전 |
| Storage | Cloud Storage Buckets (스토리지 버킷) | 0개 | 🟢 안전 |
| Database | Cloud SQL Instances (데이터베이스) | API 비활성화 | ⚪ 없음 |
| Serverless | Cloud Run Services (서버리스 컨테이너) | API 비활성화 | ⚪ 없음 |
